# Brazilian E-Commerce — Sales & Delivery Performance Analysis
**Author:** Fariba Kazi

**Business question:** Which product categories, regions, and sellers drive the most revenue — and where is delivery performance hurting customer satisfaction?

**Pipeline:** Excel (profiling) → Python/Pandas (clean + join 7 tables) → SQL → Power BI

In [2]:
import pandas as pd
orders    = pd.read_csv("olist_orders_dataset.csv")
items     = pd.read_csv("olist_order_items_dataset.csv")
products  = pd.read_csv("olist_products_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
sellers   = pd.read_csv("olist_sellers_dataset.csv")
reviews   = pd.read_csv("olist_order_reviews_dataset.csv")
category  = pd.read_csv("product_category_name_translation.csv")

In [3]:
orders_geo = orders.merge(customers, on="customer_id", how="left")

In [4]:
print("orders before:", orders.shape)
print("orders_geo    :", orders_geo.shape)
orders_geo[["order_id", "customer_id", "customer_state", "customer_city"]].head()

orders before: (99441, 8)
orders_geo    : (99441, 12)


,order_id,customer_id,customer_state,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,SP,sao paulo
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,BA,barreiras
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,GO,vianopolis
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,RN,sao goncalo do amarante
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,SP,santo andre


In [5]:
items_cat = (
    items
    .merge(products, on="product_id", how="left")
    .merge(category, on="product_category_name", how="left")
)

In [6]:
print("items    :", items.shape)
print("items_cat:", items_cat.shape)
items_cat[["order_id", "product_id", "price", "freight_value",
           "product_category_name", "product_category_name_english"]].head()

items    : (112650, 7)
items_cat: (112650, 16)


,order_id,product_id,price,freight_value,product_category_name,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,58.90,13.29,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,239.90,19.93,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,199.00,17.87,moveis_decoracao,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,12.99,12.79,perfumaria,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,199.90,18.14,ferramentas_jardim,garden_tools


In [7]:
revenue_by_category = (
    items_cat
    .groupby("product_category_name_english")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(revenue_by_category)

product_category_name_english
health_beauty            1258681.34
watches_gifts            1205005.68
bed_bath_table           1036988.68
sports_leisure            988048.97
computers_accessories     911954.32
furniture_decor           729762.49
cool_stuff                635290.85
housewares                632248.66
auto                      592720.11
garden_tools              485256.46
Name: price, dtype: float64


In [8]:
items_region = items.merge(
    orders_geo[["order_id", "customer_state"]],
    on="order_id", how="left"
)

revenue_by_state = (
    items_region
    .groupby("customer_state")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(revenue_by_state)

customer_state
SP    5202955.05
RJ    1824092.67
MG    1585308.03
RS     750304.02
PR     683083.76
SC     520553.34
BA     511349.99
DF     302603.94
GO     294591.95
ES     275037.31
Name: price, dtype: float64


In [9]:
revenue_by_seller = items.groupby("seller_id")["price"].sum().sort_values(ascending=False)

print("Top 10 sellers by revenue:")
print(revenue_by_seller.head(10))

total_sellers = revenue_by_seller.shape[0]
top10_share = revenue_by_seller.head(10).sum() / revenue_by_seller.sum() * 100
print(f"\nTotal sellers: {total_sellers}")
print(f"Top 10 sellers = {top10_share:.1f}% of all revenue")

Top 10 sellers by revenue:
seller_id
4869f7a5dfa277a7dca6462dcf3b52b2    229472.63
53243585a1d6dc2643021fd1853d8905    222776.05
4a3ca9315b744ce9f8e9374361493884    200472.92
fa1c13f2614d7b5c4749cbc52fecda94    194042.03
7c67e1448b00f6e969d365cea6b010ab    187923.89
7e93a43ef30c4f03f38b393420bc753a    176431.87
da8622b14eb17ae2831f4ac5b9dab84a    160236.57
7a67c85e85bb2ce8582c35f2203ad736    141745.53
1025f0e2d44d7041d6cf58b6550e0bfa    138968.55
955fee9216a65b617aa5c0531780ce60    135171.70
Name: price, dtype: float64

Total sellers: 3095
Top 10 sellers = 13.1% of all revenue


In [10]:
date_cols = ["order_purchase_timestamp",
             "order_delivered_customer_date",
             "order_estimated_delivery_date"]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["delay_days"]    = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days
orders["is_late"]       = orders["delay_days"] > 0

In [11]:
orders_reviews = orders.merge(reviews[["order_id", "review_score"]], on="order_id", how="left")

delivered = orders_reviews[orders_reviews["order_delivered_customer_date"].notna()]
avg_score = delivered.groupby("is_late")["review_score"].mean().round(2)
print(avg_score)

is_late
False    4.29
True     2.27
Name: review_score, dtype: float64


In [12]:
# clean order-level table: region + delivery metrics + review score
orders_clean = (
    orders
    .merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
    .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
)
orders_clean.to_csv("orders_clean.csv", index=False)

# clean item-level table: revenue + English category + seller
items_clean = items_cat[["order_id", "product_id", "seller_id",
                         "price", "freight_value",
                         "product_category_name_english"]]
items_clean.to_csv("items_clean.csv", index=False)

print("Saved:", orders_clean.shape, items_clean.shape)

Saved: (99992, 13) (112650, 6)


In [13]:
# keep only one review per order
reviews_unique = reviews.drop_duplicates(subset="order_id", keep="first")

orders_clean = (
    orders
    .merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")
    .merge(reviews_unique[["order_id", "review_score"]], on="order_id", how="left")
)
orders_clean.to_csv("orders_clean.csv", index=False)
print("orders_clean:", orders_clean.shape)

orders_clean: (99441, 13)


## SQL — validating the findings
The same questions, re-answered in SQL against a SQLite database – confirming the Python results.

In [14]:
import sqlite3

conn = sqlite3.connect("ecommerce.db")
orders_clean.to_sql("orders", conn, if_exists="replace", index=False)
items_clean.to_sql("items",  conn, if_exists="replace", index=False)
print("Loaded tables: orders, items")

Loaded tables: orders, items


In [15]:
query = """
SELECT product_category_name_english AS category,
       ROUND(SUM(price), 2) AS revenue
FROM items
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,category,revenue
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
5,furniture_decor,729762.49
6,cool_stuff,635290.85
7,housewares,632248.66
8,auto,592720.11
9,garden_tools,485256.46


In [16]:
query = """
SELECT o.customer_state AS state,
       ROUND(SUM(i.price), 2) AS revenue
FROM items i
JOIN orders o ON i.order_id = o.order_id
GROUP BY state
ORDER BY revenue DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,state,revenue
0,SP,5202955.05
1,RJ,1824092.67
2,MG,1585308.03
3,RS,750304.02
4,PR,683083.76
5,SC,520553.34
6,BA,511349.99
7,DF,302603.94
8,GO,294591.95
9,ES,275037.31


In [17]:
query = """
SELECT is_late,
       ROUND(AVG(review_score), 2) AS avg_score,
       COUNT(*) AS num_orders
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
GROUP BY is_late;
"""
pd.read_sql_query(query, conn)

,is_late,avg_score,num_orders
0,0,4.29,89941
1,1,2.27,6535


In [19]:
dashboard_data = items_clean.merge(
    orders_clean[["order_id", "customer_state", "is_late", "review_score", "delivery_days"]],
    on="order_id", how="left"
)
dashboard_data.to_csv("dashboard_data.csv", index=False)
print("done")

done


## Key Findings
1. **Top categories:** Health & Beauty (BRL 1.26M), Watches & Gifts (BRL 1.20M), and Bed/Bath/Table (BRL 1.04M) lead revenue.
2. **Regional concentration:** São Paulo dominates customers (about 42%) and revenue (BRL 5.2M, nearly 3× the next state); the top 3 states are all in the Southeast.
3. **Seller concentration:** Of 3,095 sellers, the top 10 (0.3%) drive 13.1% of revenue.
4. **Delivery drives satisfaction:** On-time orders average 4.29 stars vs 2.27 for late orders — late delivery nearly halves the score.

## Recommendation
Delivery reliability is the highest-leverage fix: since late orders cut satisfaction nearly in half, tightening logistics or setting more realistic delivery estimates would protect reviews and retention.